# Kiwi — reconocer herramientas desde el celular

EmbeddingGemma 2 + Gradio: una página para subir una foto y una API `identify`.
Presentar **un objeto por foto**, completo, sobre fondo limpio. No requiere entrenamiento.

1. En **Entorno de ejecución → Cambiar tipo de entorno**, seleccionar GPU si está disponible; CPU también sirve para probar, con mayor latencia.
2. Ejecutar las celdas 1–5 en orden. La carga inicial descarga el modelo.
3. Ejecutar la celda 6, introducir una contraseña y abrir en el celular el enlace `gradio.live` que aparezca.
4. Para mejorar las comparaciones, añadir fotos reales con la celda opcional de referencias.

El enlace es temporal: desaparece si Colab termina la sesión. Colab gratuito restringe interactuar principalmente por una interfaz web externa; usar este notebook para pruebas interactivas y revisar las condiciones de Colab. Un plan de pago con saldo no garantiza disponibilidad permanente.

Hay **11 candidatos** en la lista actual; el reto exige elegir **9**. Antes de integrar LEDs, ajustar la lista y definir sus ubicaciones reales.

Fuentes: [Google: modelo y ejemplos](https://ai.google.dev/gemma/docs/embeddinggemma/multimodal-embeddinggemma-with-sentence-transformers), [Gradio: enlaces y API](https://gradio.app/guides/sharing-your-app), [Colab: condiciones](https://research.google.com/colaboratory/faq.html).

## 1. Instalar dependencias
Si Colab pide reiniciar la sesión después de instalar, reiniciar y continuar con la celda 2. No reinstalamos PyTorch: usamos el de Colab.

In [ ]:
%pip install -q -U sentence-transformers transformers gradio


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configurar categorías y cargar el modelo
Las descripciones de cable, capacitores y soldadura deben ajustarse a los objetos reales. Los IDs son provisionales; no representan todavía posiciones físicas.

`MIN_SIMILARITY` y `MIN_MARGIN` quedan sin configurar: inicialmente veremos candidatos, pero `recognized` será falso. Ajustarlos solo después de pruebas con objetos conocidos, desconocidos y la mesa vacía. La similitud **no es un porcentaje de confianza**.

In [ ]:
import os
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
import time
import numpy as np
import torch
from PIL import Image, ImageOps
from sentence_transformers import SentenceTransformer

MODEL_ID = 'google/embeddinggemma-2'
CATEGORIES = {
    'soldering_station': 'An electronics soldering and hot air rework station with a control unit and iron holder',
    'digital_multimeter': 'A handheld digital multimeter with a screen and rotary selector',
    'bench_power_supply': 'A laboratory bench DC power supply with output terminals and controls',
    'servo_motor': 'A small blue micro servo motor with a three-wire cable',
    'oscilloscope': 'A digital oscilloscope with a waveform screen and input connectors',
    'function_generator': 'A laboratory function generator with frequency controls and signal outputs',
    'variable_capacitor': 'A laboratory variable parallel plate capacitor with circular metal plates on a base',
    'capacitors': 'A small cylindrical electrolytic capacitor with two wire leads',
    'power_adapter': 'A rectangular AC DC power adapter with an integrated output cable',
    'multimeter_box': 'A yellow orange cardboard retail box for an AstroAI digital multimeter',
    'solder': 'An electric soldering iron with a handle and pointed metal tip',
}
MIN_SIMILARITY = None
MIN_MARGIN = None
LOCATION_BY_CATEGORY = {}  # Ejemplo al definir posiciones reales: {'digital_multimeter': 2}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
print('Dispositivo:', device, '| precisión:', dtype)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Cargando el modelo; la primera vez puede tardar varios minutos...')
model = SentenceTransformer(
    MODEL_ID,
    device=device,
    config_kwargs={'audio_config': None},
    model_kwargs={'torch_dtype': dtype},
)
model.eval()
print('Modelo cargado:', MODEL_ID)
print('Categorías:', len(CATEGORIES), '(seleccionar 9 antes de integrar los LEDs)')

## 3. Decidir cuándo aceptar una comparación
Rechazamos resultados débiles o ambiguos. Si faltan umbrales de validación, mostramos el candidato sin autorizar la señalización.

In [ ]:
import math

def decide(scores, minimum=None, margin=None):
    ranking = sorted(
        ((name, float(score)) for name, score in scores.items() if math.isfinite(float(score))),
        key=lambda item: item[1], reverse=True,
    )
    result = {'candidate': None, 'recognized': False, 'similarity': None,
              'margin': None, 'reason': 'No hay suficientes categorías válidas.'}
    if not ranking:
        return result
    result.update(candidate=ranking[0][0], similarity=ranking[0][1])
    if len(ranking) < 2:
        return result
    result['margin'] = ranking[0][1] - ranking[1][1]
    if minimum is None or margin is None:
        result['reason'] = 'Modo de prueba: umbrales sin calibrar.'
    elif ranking[0][1] < minimum:
        result['reason'] = 'Similitud insuficiente: objeto desconocido o captura inadecuada.'
    elif result['margin'] < margin:
        result['reason'] = 'Comparación ambigua entre las categorías principales.'
    else:
        result.update(recognized=True, reason='Comparación aceptada con los umbrales configurados.')
    return result

## 4. Preparar la comparación y la respuesta de la API
El modo inicial compara la imagen con las descripciones. Si cargamos referencias en la celda 5, compara con fotografías reales. Cambiar de modo requiere volver a validar los umbrales.

In [ ]:
names = list(CATEGORIES)
text_vectors = model.encode(list(CATEGORIES.values()), normalize_embeddings=True, convert_to_numpy=True)
reference_vectors = None
reference_labels = []

def prepare_image(image):
    image = ImageOps.exif_transpose(image).convert('RGB')
    image.thumbnail((1024, 1024))
    return image

def identify(image):
    if image is None:
        return {'recognized': False, 'candidate': None, 'location_id': None,
                'reason': 'Sube una fotografía.'}
    start = time.perf_counter()
    vector = model.encode({'image': prepare_image(image)}, normalize_embeddings=True, convert_to_numpy=True)
    if reference_vectors is None:
        raw_scores = np.asarray(text_vectors @ vector).reshape(-1)
        scores = dict(zip(names, raw_scores))
        mode = 'image_to_text'
    else:
        raw_scores = np.asarray(reference_vectors @ vector).reshape(-1)
        scores = {name: max(float(score) for label, score in zip(reference_labels, raw_scores) if label == name)
                  for name in names}
        mode = 'image_to_image'
    result = decide(scores, MIN_SIMILARITY, MIN_MARGIN)
    result['mode'] = mode
    result['location_id'] = LOCATION_BY_CATEGORY.get(result['candidate']) if result['recognized'] else None
    result['elapsed_seconds'] = round(time.perf_counter() - start, 3)
    result['ranking'] = [{'object': name, 'similarity': round(float(score), 4)}
                         for name, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)]
    return result

print('Comparación preparada. Las referencias y los umbrales se pueden ajustar sin recargar el modelo.')

## 5. Cargar fotos de referencia desde Drive

Guardar `dataset_prepared/references.zip` en `Mi unidad/Kiwi/references.zip`. Esta celda lee el archivo desde Drive y calcula los vectores de las 41 referencias. Mantener `LOAD_REFERENCE_ZIP = True` para activar la comparación imagen contra imagen. Las 3 fotos de `evaluation.zip` se reservan para pruebas.

Los vectores quedan en memoria durante la sesión; este notebook todavía no genera ni guarda `reference_embeddings.npz`. Al cambiar el ZIP hay que ejecutar de nuevo esta celda. Volver a ejecutar la sección 4 limpia las referencias anteriores.


In [ ]:
LOAD_REFERENCE_ZIP = True
if LOAD_REFERENCE_ZIP:
    import io
    import zipfile
    from pathlib import PurePosixPath, Path
    zip_path = Path('/content/drive/MyDrive/Kiwi/references.zip')
    if not zip_path.is_file():
        available = [p.name for p in zip_path.parent.glob('*.zip')]
        raise FileNotFoundError(f'No existe {zip_path}. ZIPs disponibles en Kiwi: {available}')
    zip_name, zip_data = zip_path.name, zip_path.read_bytes()
    print('Cargando referencias desde Drive:', zip_name)
    if len(zip_data) > 100 * 1024 * 1024:
        raise ValueError('El ZIP debe pesar menos de 100 MB.')
    vectors, labels = [], []
    with zipfile.ZipFile(io.BytesIO(zip_data)) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) > 500 * 1024 * 1024:
            raise ValueError('El contenido del ZIP supera 500 MB.')
        for member in members:
            path = PurePosixPath(member.filename)
            if member.is_dir() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
                continue
            if '__MACOSX' in path.parts or path.name.startswith('.'):
                continue
            label = path.parent.name
            if label not in CATEGORIES:
                raise ValueError(f'Carpeta desconocida: {label}. Usar los nombres de CATEGORIES.')
            if member.file_size > 15 * 1024 * 1024:
                raise ValueError(f'Foto demasiado grande: {path.name}. Máximo 15 MB.')
            with Image.open(io.BytesIO(archive.read(member))) as photo:
                vector = model.encode({'image': prepare_image(photo)}, normalize_embeddings=True, convert_to_numpy=True)
            vectors.append(vector)
            labels.append(label)
    missing = set(CATEGORIES) - set(labels)
    if missing:
        raise ValueError('Faltan referencias de: ' + ', '.join(sorted(missing)))
    reference_vectors = np.stack(vectors)
    reference_labels = labels
    MIN_SIMILARITY = None
    MIN_MARGIN = None
    print('Referencias cargadas:', {name: labels.count(name) for name in names})
    print('Se restablecieron los umbrales: calibrar para el nuevo modo.')
else:
    print('Sin cambios en las referencias. Modo actual:', 'fotografías' if reference_vectors is not None else 'descripciones')

## 6. Abrir la página del celular y la API
Ejecutar esta celda e introducir una contraseña para esta sesión. El usuario será `kiwi`.
Abrir el enlace **https://…gradio.live** desde el celular y seleccionar una foto de la galería o usar la cámara cuando el navegador lo permita. No requiere que el celular y la computadora estén en la misma red.

El acceso es temporal y pasa por el servicio de enlaces de Gradio. No usar fotos sensibles. Mantener la sesión de Colab activa; cerrar el servicio con `demo.close()` al terminar.

La página devuelve candidatos y similitudes. Al inicio `recognized=false` es intencional: no se ha validado la aceptación. No hay LEDs conectados en este notebook.

In [ ]:
import getpass
import gradio as gr

if 'demo' in globals():
    demo.close()
APP_USER = 'kiwi'
APP_PASSWORD = getpass.getpass('Contraseña para la página y la API (mínimo 12 caracteres): ')
if len(APP_PASSWORD) < 12:
    raise ValueError('Usa al menos 12 caracteres y vuelve a ejecutar esta celda.')
demo = gr.Interface(
    fn=identify,
    inputs=gr.Image(type='pil', sources=['upload', 'webcam'], label='Fotografía de una herramienta'),
    outputs=gr.JSON(label='Resultado'),
    title='Kiwi · ¿Dónde guardo esta herramienta?',
    description='Presenta un objeto por foto. El resultado inicial es un candidato de prueba; no es una certeza.',
    api_name='identify',
    flagging_mode='never',
    submit_btn='Identificar',
    clear_btn='Limpiar',
)
demo.queue(default_concurrency_limit=1, max_size=10)
demo.launch(share=True, auth=(APP_USER, APP_PASSWORD), debug=False)

## 7. Consumir la API desde otra computadora o aplicación
La página del celular ya envía la foto a la API. No hace falta escribir un cliente para probarla.
Para una integración propia, usar **Use via API** en el pie de la página y consultar el esquema generado. Gradio utiliza su protocolo de carga y cola; `identify` no es una ruta REST simple para enviar JPEG directamente.

Ejemplo de cliente Python (ejecutar en la computadora cliente):
```python
from getpass import getpass
from gradio_client import Client, handle_file

client = Client('https://TU-ENLACE.gradio.live', auth=('kiwi', getpass('Contraseña: ')))
result = client.predict(handle_file('foto.jpg'), api_name='/identify')
print(result)
```
Instalar `gradio_client` en el cliente. No guardar contraseñas en el notebook ni en código compartido.

## Validación antes de conectar luces
- Elegir las **9 categorías** definitivas y modificar `CATEGORIES`; ejecutar otra vez desde la celda 4.
- Probar fotografías nuevas, mesa vacía y objetos ajenos. Registrar candidatos, similitudes y confusiones.
- Ajustar `MIN_SIMILARITY` y `MIN_MARGIN` para el modo que se utilizará, ejecutando la celda 2 únicamente si se quiere recargar todo. También se pueden asignar estas variables en una nueva celda sin recargar el modelo.
- Definir `LOCATION_BY_CATEGORY` con las ubicaciones reales.
- El cliente debe activar luces solo cuando `recognized` sea verdadero y `location_id` tenga un valor válido. Ante rechazo o error, apagar la señalización anterior.

## Problemas frecuentes
- **Modelo desconocido:** ejecutar instalación; reiniciar sesión si quedaron librerías antiguas en memoria.
- **Acceso al modelo denegado:** revisar la página oficial en Hugging Face. Si requiere autenticación, usar un secreto `HF_TOKEN` de Colab; no pegar tokens en celdas compartidas.
- **Sin memoria GPU:** reducir el tamaño de las fotos o usar CPU; no cambiar a `float16`, que el modelo no admite.
- **Enlace dejó de funcionar:** comprobar que la sesión siga conectada y volver a ejecutar la celda 6. El enlace puede cambiar.
- **Dos equipos se confunden:** mejorar encuadre y fotos de referencia; no activar LEDs con una diferencia pequeña entre candidatos.